In [36]:
import pandas as pd
import numpy as np

# -------------------------
# Mock Training Data
# -------------------------
X_train = pd.DataFrame({
    "tenure": [1, 5, 10, 15, 20, 24, 30, 36, 40, 48],
    "MonthlyCharges": [70, 65, 90, 85, 50, 45, 100, 95, 55, 40],
    "TotalCharges": [70, 325, 900, 1275, 1000, 1080, 3000, 3420, 2200, 1920],
    "num_support_tickets": [5, 4, 3, 2, 1, 1, 0, 0, 1, 0]
})

# Churn labels
y_train = np.array([
    1,  # churned
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0
], dtype=np.float32)

# -------------------------
# Mock Test Data
# -------------------------
X_test = pd.DataFrame({
    "tenure": [3, 18, 42],
    "MonthlyCharges": [75, 60, 45],
    "TotalCharges": [225, 1080, 1890],
    "num_support_tickets": [4, 2, 0]
})

print("Training Features:")
print(X_train)

print("\nTraining Labels:")
print(y_train)

print("\nTest Features:")
print(X_test)

Training Features:
   tenure  MonthlyCharges  TotalCharges  num_support_tickets
0       1              70            70                    5
1       5              65           325                    4
2      10              90           900                    3
3      15              85          1275                    2
4      20              50          1000                    1
5      24              45          1080                    1
6      30             100          3000                    0
7      36              95          3420                    0
8      40              55          2200                    1
9      48              40          1920                    0

Training Labels:
[1. 1. 1. 0. 0. 0. 0. 0. 0. 0.]

Test Features:
   tenure  MonthlyCharges  TotalCharges  num_support_tickets
0       3              75           225                    4
1      18              60          1080                    2
2      42              45          1890                    0


In [49]:
import torch.nn as nn
import torch

class ChurnClassifier(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, hidden_dim_02=8):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),  # <-- 4 not 20
            nn.ReLU(),

            nn.Linear(hidden_dim, hidden_dim_02),
            nn.ReLU(),

            nn.Linear(hidden_dim_02, 1),
            nn.Sigmoid()
        )

    def forward(self, X):
        return self.net(X.float())

In [52]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from skorch import NeuralNetBinaryClassifier

# 1. Define which columns get scaled
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# 2. Setup the preprocessing step (leaves other columns untouched)
preprocessor = ColumnTransformer(
    transformers=[('num', MinMaxScaler(), num_cols)],
    remainder='passthrough'
)

# 3. Wrap the PyTorch model using Skorch
# Note: Skorch handles the training loop, epochs, and optimizer under the hood!
pytorch_skorch_wrapper = NeuralNetBinaryClassifier(
    module=ChurnClassifier,
    module__input_dim=4,          # Passes this argument to ChurnClassifier's __init__
    criterion=nn.BCEWithLogitsLoss, # Standard loss for binary classification
    optimizer=torch.optim.Adam,
    lr=0.001,
    max_epochs=20,
    batch_size=64,
    train_split=None               # Disables skorch's internal validation split if you want to use sklearn's
)

# 4. Bind them together into one flawless Pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('nn_model', pytorch_skorch_wrapper)
])

In [53]:
# Assuming X_train is a DataFrame containing raw columns (tenure, etc.) 
# and y_train is a NumPy array of 0s and 1s (cast to float32 for PyTorch)
import numpy as np
y_train = y_train.astype(np.float32)

# Train the entire pipeline (Scales data -> Passes to PyTorch -> Runs training epochs)
pipeline.fit(X_train, y_train)

# Make predictions on RAW test/production data 
# (The pipeline scales the data automatically before passing it to PyTorch)
predictions = pipeline.predict(X_test)
probabilities = pipeline.predict_proba(X_test)

  epoch    train_loss     dur
-------  ------------  ------
      1        0.8336  0.0036
      2        0.8330  0.0050
      3        0.8325  0.0065
      4        0.8320  0.0076
      5        0.8315  0.0033
      6        0.8310  0.0039
      7        0.8305  0.0032
      8        0.8300  0.0026
      9        0.8295  0.0028
     10        0.8290  0.0027
     11        0.8285  0.0027
     12        0.8281  0.0031
     13        0.8275  0.0058
     14        0.8270  0.0038
     15        0.8265  0.0032
     16        0.8260  0.0042
     17        0.8255  0.0045
     18        0.8250  0.0044
     19        0.8245  0.0030
     20        0.8240  0.0031


In [54]:
print(predictions)

[1 1 1]


In [55]:
print(probabilities)

[[0.3652926  0.6347074 ]
 [0.3726136  0.6273864 ]
 [0.37434667 0.6256533 ]]
